In [1]:
import json
import pandas as pd
import numpy as np
import glob
import os
import re

metadata_files = glob.glob("*.json")
# metadata_files = glob.glob("metadata.json")
gps_samples, ypr_samples = [], []

for file in metadata_files:
    with open(file, "r") as f:
        data = json.load(f)

    for stream_id, stream_data in data.items():
        if isinstance(stream_data, dict) and "streams" in stream_data:
            streams = stream_data["streams"]

            # === GPS ===
            gps_stream = streams.get("GPS5")
            if gps_stream and "samples" in gps_stream:
                for s in gps_stream["samples"]:
                    val = s.get("value", [])
                    if len(val) >= 3:
                        gps_samples.append({
                            "file": file,
                            "stream_id": stream_id,
                            "cts": s.get("cts"),
                            "datetime": s.get("date"),
                            "lat": val[0],
                            "lon": val[1],
                            "alt": val[2] if len(val) > 2 else None,
                            "spd_3d": val[3] if len(val) > 3 else None,
                            "spd_2d": val[4] if len(val) > 4 else None
                        })

            # === Yaw / Pitch / Roll ===
            for key in ["CORI", "GYRO", "ISOG", "CAMERA_ORIENTATION"]:
                if key in streams:
                    ypr_stream = streams[key]
                    if ypr_stream and "samples" in ypr_stream:
                        for s in ypr_stream["samples"]:
                            val = s.get("value", [])
                            if len(val) >= 3:
                                ypr_samples.append({
                                    "file": file,
                                    "stream_id": stream_id,
                                    "cts": s.get("cts"),
                                    "datetime": s.get("date"),
                                    "yaw": val[0],
                                    "pitch": val[1],
                                    "roll": val[2]
                                })

# === Buat DataFrame ===
gps_df = pd.DataFrame(gps_samples)
ypr_df = pd.DataFrame(ypr_samples)

# --- convert datetime ---
gps_df["datetime"] = pd.to_datetime(gps_df["datetime"], errors="coerce")
ypr_df["datetime"] = pd.to_datetime(ypr_df["datetime"], errors="coerce")

# --- waktu relatif (detik dari awal GPS) ---
if not gps_df.empty:
    start_time = gps_df["datetime"].min()
    gps_df["t_rel_sec"] = (gps_df["datetime"] - start_time).dt.total_seconds()

if not ypr_df.empty:
    ypr_df["t_rel_sec"] = (ypr_df["datetime"] - start_time).dt.total_seconds()

# --- interpolasi YPR ke timeline GPS ---
ypr_interp = {}
for col in ["yaw", "pitch", "roll"]:
    if not ypr_df.empty:
        ypr_interp[col] = np.interp(
            gps_df["t_rel_sec"],
            ypr_df["t_rel_sec"],
            ypr_df[col]
        )
    else:
        ypr_interp[col] = np.nan

for k, v in ypr_interp.items():
    gps_df[k] = v

# === Tambah placeholder sensor lain (biar seragam) ===
for col in ["gyro_x","gyro_y","gyro_z","accel_x","accel_y","accel_z","temp"]:
    gps_df[col] = np.nan

# === Simpan hasil gabungan ===
gps_df.to_csv("gps_ypr_merged.csv", index=False)
print(f"✅ Gabungan selesai — total {len(gps_df)} baris, disimpan ke gps_ypr_merged.csv")
print(gps_df.head(10))


✅ Gabungan selesai — total 1054 baris, disimpan ke gps_ypr_merged.csv
             file stream_id         cts                         datetime  \
0  metadata2.json         1  164.660000 2025-11-11 02:49:47.695000+00:00   
1  metadata2.json         1  219.647944 2025-11-11 02:49:47.798000+00:00   
2  metadata2.json         1  274.635889 2025-11-11 02:49:47.902000+00:00   
3  metadata2.json         1  329.623833 2025-11-11 02:49:48.006000+00:00   
4  metadata2.json         1  384.611778 2025-11-11 02:49:48.110000+00:00   
5  metadata2.json         1  439.599722 2025-11-11 02:49:48.214000+00:00   
6  metadata2.json         1  494.587667 2025-11-11 02:49:48.318000+00:00   
7  metadata2.json         1  549.575611 2025-11-11 02:49:48.422000+00:00   
8  metadata2.json         1  604.563556 2025-11-11 02:49:48.526000+00:00   
9  metadata2.json         1  659.551500 2025-11-11 02:49:48.630000+00:00   

        lat         lon      alt  spd_3d  spd_2d  t_rel_sec       yaw  \
0 -6.917102  107.609

In [2]:
import os
import re
import pandas as pd
import numpy as np

# =========================
# PARAMETER
# =========================
frames_dir = r"D:\metadata\360\single_file\360\frame_360\5\GS014251"                      # folder frame hasil extract video 360
merged_sensor_csv = "gps_ypr_merged.csv"  # sensor timeline
output_csv = "frames_mapping_fullsensor_with_ypr.csv"

video_duration = 57.9  # durasi video (detik)

# =========================
# LOAD SENSOR DATA
# =========================
gps_df = pd.read_csv(merged_sensor_csv)

gps_df["datetime"] = pd.to_datetime(
    gps_df["datetime"], errors="coerce"
)

gps_df = gps_df.sort_values("t_rel_sec").reset_index(drop=True)

t_min = gps_df["t_rel_sec"].min()
t_max = gps_df["t_rel_sec"].max()

# =========================
# LOAD FRAME LIST
# =========================
img_files = sorted(
    f for f in os.listdir(frames_dir)
    if re.match(r"^GS\d+_image_\d+\.jpg$", f)
)

if not img_files:
    raise SystemExit("Tidak ditemukan frame GS*_image_xxxx.jpg")

print(f"Total frame: {len(img_files)}")

# =========================
# TIMELINE VIDEO (FRAME DOMAIN)
# =========================
timestamps = np.linspace(0, video_duration, len(img_files))

# =========================
# MAPPING PER FRAME
# =========================
mapping_rows = []

sensor_cols = [
    "lat", "lon", "alt",
    "spd_3d", "spd_2d",
    "yaw", "pitch", "roll",
    "gyro_x", "gyro_y", "gyro_z",
    "accel_x", "accel_y", "accel_z",
    "temp"
]

for img_name, t_vid in zip(img_files, timestamps):

    # jaga supaya tidak keluar domain sensor
    t_clipped = np.clip(t_vid, t_min, t_max)

    # === INTERPOLASI NUMERIK ===
    interpolated = {}
    for col in sensor_cols:
        interpolated[col] = np.interp(
            t_clipped,
            gps_df["t_rel_sec"],
            gps_df[col]
        )

    # === DATETIME TERDEKAT (TIDAK DIINTERPOLASI) ===
    nearest_idx = (
        gps_df["t_rel_sec"]
        .sub(t_clipped)
        .abs()
        .idxmin()
    )

    base_datetime = gps_df.loc[nearest_idx, "datetime"]

    # === ROW FINAL ===
    row = {
        "image_name": img_name,
        "timestamp_video": t_vid,
        "datetime": (
            base_datetime.isoformat()
            if pd.notna(base_datetime)
            else None
        ),
    }

    row.update(interpolated)
    mapping_rows.append(row)

# =========================
# SAVE CSV
# =========================
mapping_df = pd.DataFrame(mapping_rows)
mapping_df.to_csv(output_csv, index=False)

print(f"Mapping selesai: {len(mapping_df)} frame")
print(mapping_df.head(5))


Total frame: 289
Mapping selesai: 289 frame
                image_name  timestamp_video                          datetime  \
0  GS014251_image_0001.jpg         0.000000  2025-11-11T02:49:47.695000+00:00   
1  GS014251_image_0002.jpg         0.201042  2025-11-11T02:49:47.902000+00:00   
2  GS014251_image_0003.jpg         0.402083  2025-11-11T02:49:48.110000+00:00   
3  GS014251_image_0004.jpg         0.603125  2025-11-11T02:49:48.318000+00:00   
4  GS014251_image_0005.jpg         0.804167  2025-11-11T02:49:48.526000+00:00   

        lat         lon         alt    spd_3d    spd_2d       yaw     pitch  \
0 -6.917102  107.609204  726.328000  0.144000  0.190000  1.000000 -0.002991   
1 -6.917102  107.609204  726.361943  0.163057  0.179427  0.999931 -0.005862   
2 -6.917102  107.609204  726.340994  0.128713  0.188758  0.999833 -0.007086   
3 -6.917101  107.609204  726.362560  0.167565  0.133822  0.999705 -0.003010   
4 -6.917101  107.609204  726.318611  0.103450  0.162901  0.999576 -0.00020

In [3]:
import piexif
from PIL import Image
import pandas as pd
import numpy as np
import os
import json
from datetime import datetime
from subprocess import run

# === PATH ===
csv_path = "frames_mapping_fullsensor_with_ypr.csv"
frames_dir = r"D:\metadata\360\single_file\360\frame_360\5\GS014251"  
output_dir = "output_with_exif_metashape_safe_full"
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(csv_path)

def deg_to_dms_rational(deg_float):
    """Decimal degree → DMS rational (EXIF GPS)"""
    deg = int(abs(deg_float))
    min_float = (abs(deg_float) - deg) * 60
    minutes = int(min_float)
    seconds = int((min_float - minutes) * 60 * 100)
    return ((deg, 1), (minutes, 1), (seconds, 100))

for _, row in df.iterrows():

    img_name = row.get("image_name")
    if not img_name:
        continue

    img_path = os.path.join(frames_dir, img_name)
    if not os.path.exists(img_path):
        print(f"Skip: {img_name} tidak ditemukan")
        continue

    img = Image.open(img_path)

    # === EXIF STRUCTURE ===
    exif_dict = {
        "0th": {},
        "Exif": {},
        "GPS": {},
        "1st": {},
        "thumbnail": None
    }

    # =========================================================
    # GPS (INI YANG DIPAKAI METASHAPE)
    # =========================================================
    lat, lon, alt = row["lat"], row["lon"], row["alt"]

    if not pd.isna(lat) and not pd.isna(lon):
        exif_dict["GPS"][piexif.GPSIFD.GPSLatitudeRef] = "S" if lat < 0 else "N"
        exif_dict["GPS"][piexif.GPSIFD.GPSLatitude] = deg_to_dms_rational(lat)

        exif_dict["GPS"][piexif.GPSIFD.GPSLongitudeRef] = "W" if lon < 0 else "E"
        exif_dict["GPS"][piexif.GPSIFD.GPSLongitude] = deg_to_dms_rational(lon)

    if not pd.isna(alt):
        exif_dict["GPS"][piexif.GPSIFD.GPSAltitude] = (int(alt * 100), 100)

    # =========================================================
    # DATETIME
    # =========================================================
    try:
        dt = datetime.fromisoformat(str(row["datetime"]))
        dt_str = dt.strftime("%Y:%m:%d %H:%M:%S")
        exif_dict["0th"][piexif.ImageIFD.DateTime] = dt_str
        exif_dict["Exif"][piexif.ExifIFD.DateTimeOriginal] = dt_str
    except Exception:
        pass

    # =========================================================
    # SENSOR METADATA TAMBAHAN (AMAN, TIDAK DIPAKAI METASHAPE)
    # =========================================================
    sensor_meta = {
        "yaw_platform_deg": row.get("yaw"),
        "pitch_platform_deg": row.get("pitch"),
        "roll_platform_deg": row.get("roll"),
        "spd_3d": row.get("spd_3d"),
        "spd_2d": row.get("spd_2d"),
        "gyro": {
            "x": row.get("gyro_x"),
            "y": row.get("gyro_y"),
            "z": row.get("gyro_z"),
        },
        "accel": {
            "x": row.get("accel_x"),
            "y": row.get("accel_y"),
            "z": row.get("accel_z"),
        },
        "temp": row.get("temp"),
    }

    exif_dict["0th"][piexif.ImageIFD.ImageDescription] = (
        json.dumps(sensor_meta).encode("utf-8")
    )

    # =========================================================
    # SAVE IMAGE + EXIF
    # =========================================================
    out_path = os.path.join(output_dir, img_name)
    img.save(out_path, exif=piexif.dump(exif_dict))

    # =========================================================
    # XMP ORIENTATION (METASHAPE PRIORITY)
    # =========================================================
    yaw = row.get("yaw", np.nan)
    pitch = row.get("pitch", np.nan)
    roll = row.get("roll", np.nan)

    exiftool_cmd = ["exiftool", "-overwrite_original"]

    # METASHAPE PAKAI INI
    if not pd.isna(yaw):
        exiftool_cmd.append(f"-XMP:CameraYaw={yaw}")

    # Tambahan (aman, tidak mengganggu)
    if not pd.isna(pitch):
        exiftool_cmd.append(f"-XMP:PosePitchDegrees={pitch}")
    if not pd.isna(roll):
        exiftool_cmd.append(f"-XMP:PoseRollDegrees={roll}")

    exiftool_cmd.append(out_path)
    run(exiftool_cmd)

    print(f"OK: {img_name}")

print("🎉 Selesai. Semua frame aman & optimal untuk Metashape.")


OK: GS014251_image_0001.jpg
OK: GS014251_image_0002.jpg
OK: GS014251_image_0003.jpg
OK: GS014251_image_0004.jpg
OK: GS014251_image_0005.jpg
OK: GS014251_image_0006.jpg
OK: GS014251_image_0007.jpg
OK: GS014251_image_0008.jpg
OK: GS014251_image_0009.jpg
OK: GS014251_image_0010.jpg
OK: GS014251_image_0011.jpg
OK: GS014251_image_0012.jpg
OK: GS014251_image_0013.jpg
OK: GS014251_image_0014.jpg
OK: GS014251_image_0015.jpg
OK: GS014251_image_0016.jpg
OK: GS014251_image_0017.jpg
OK: GS014251_image_0018.jpg
OK: GS014251_image_0019.jpg
OK: GS014251_image_0020.jpg
OK: GS014251_image_0021.jpg
OK: GS014251_image_0022.jpg
OK: GS014251_image_0023.jpg
OK: GS014251_image_0024.jpg
OK: GS014251_image_0025.jpg
OK: GS014251_image_0026.jpg
OK: GS014251_image_0027.jpg
OK: GS014251_image_0028.jpg
OK: GS014251_image_0029.jpg
OK: GS014251_image_0030.jpg
OK: GS014251_image_0031.jpg
OK: GS014251_image_0032.jpg
OK: GS014251_image_0033.jpg
OK: GS014251_image_0034.jpg
OK: GS014251_image_0035.jpg
OK: GS014251_image_0